In [2]:
import json

config = {
    "pipeline": "sales_etl",
    "source": {"type": "BigQuery", "dataset": "raw_sales"},
    "target": {"type": "MySQL", "table": "sales_fact"},
    "schedule": "0 2 * * *",
    "retries": 3
}

with open("config.json","w") as outfile:
    json.dump(config,outfile,indent=2)
    
with open("config.json","r") as infile:
    data=json.load(infile)
    
print(data["target"]["table"])

sales_fact


In [1]:
# Parse a JSON string from an API Easy
# An API returns the string below. Use json.loads() to parse it, then extract and print: pipeline name, total rows, and each step's name and status.
# api_response = '{"pipeline":"finance_load","total_rows":5400,"steps":[{"name":"extract","status":"ok"},{"name":"transform","status":"ok"},{"name":"load","status":"failed"}]}'


import json

api_response = '{"pipeline":"finance_load","total_rows":5400,"steps":[{"name":"extract","status":"ok"},{"name":"transform","status":"ok"},{"name":"load","status":"failed"}]}'

#parse json string
data=json.loads(api_response)

# extract amd print required values

In [4]:
print(data["pipeline"])
print(data["total_rows"])

for steps in data["steps"]:
    print(f"name: {steps['name']},status: {steps['status']}")

finance_load
5400
name: extract,status: ok
name: transform,status: ok
name: load,status: failed


In [10]:

# Nested JSON — flatten to rows Medium
# Each order has nested line items. Flatten the structure so each line item becomes its own row (like a database table), inheriting the order-level fields.
# orders = [
#   {"order_id":"ORD001","customer":"Riya","date":"2024-06-01",
#    "items":[{"sku":"SKU-A","qty":2,"price":500},
#             {"sku":"SKU-B","qty":1,"price":1200}]},
#   {"order_id":"ORD002","customer":"Arjun","date":"2024-06-01",
#    "items":[{"sku":"SKU-C","qty":5,"price":200}]}
# ]

orders = [
    {
        "order_id": "ORD001",
        "customer": "Riya",
        "date": "2024-06-01",
        "items": [
            {"sku": "SKU-A", "qty": 2, "price": 500},
            {"sku": "SKU-B", "qty": 1, "price": 1200}
        ]
    },
    {
        "order_id": "ORD002",
        "customer": "Arjun",
        "date": "2024-06-01",
        "items": [
            {"sku": "SKU-C", "qty": 5, "price": 200}
        ]
    }
]

rows = []

for order in orders:
    for item in order["items"]:
        rows.append({
            "order_id":order["order_id"],
            "customer":order["customer"],
            "date":order["date"],
            "sku": item["sku"],
            "qty":item["qty"],
            "price":item["price"]
        })

for row in rows:
    print(row)

{'order_id': 'ORD001', 'customer': 'Riya', 'date': '2024-06-01', 'sku': 'SKU-A', 'qty': 2, 'price': 500}
{'order_id': 'ORD001', 'customer': 'Riya', 'date': '2024-06-01', 'sku': 'SKU-B', 'qty': 1, 'price': 1200}
{'order_id': 'ORD002', 'customer': 'Arjun', 'date': '2024-06-01', 'sku': 'SKU-C', 'qty': 5, 'price': 200}


In [ ]:

# J4
# JSON ↔ CSV conversion Medium
# Load the orders JSON from J3, flatten it (as above), and write it to a CSV using csv.DictWriter. Then read the CSV back and reconstruct the JSON structure. This is a core ETL pattern.
import json
import csv 

orders = [
    {
        "order_id": "ORD001",
        "customer": "Riya",
        "date": "2024-06-01",
        "items": [
            {"sku": "SKU-A", "qty": 2, "price": 500},
            {"sku": "SKU-B", "qty": 1, "price": 1200}
        ]
    },
    {
        "order_id": "ORD002",
        "customer": "Arjun",
        "date": "2024-06-01",
        "items": [
            {"sku": "SKU-C", "qty": 5, "price": 200}
        ]
    }
]

# -----------------------------
# Step 1: Flatten JSON
# -----------------------------

rows = []

for order in orders:
    for item in order["items"]:
        rows.append({
            "order_id":order["order_id"],
            "customer":order["customer"],
            "date":order["date"],
            "sku": item["sku"],
            "qty":item["qty"],
            "price":item["price"]
        })

for row in rows:
    print(row)
    
# -----------------------------
# Step 2: Write to CSV
# -----------------------------

csv_file="orders.csv"

with open(csv_file,"w",newline="") as file:
    fieldnames=["order_id","customer","date","sku","qty","price"]
    writer=csv.DictWriter(file,fieldnames=fieldnames)
    
    writer.writeheader()
    writer.writerows(rows)
    
print("CSV file created successfully.")

{'order_id': 'ORD001', 'customer': 'Riya', 'date': '2024-06-01', 'sku': 'SKU-A', 'qty': 2, 'price': 500}
{'order_id': 'ORD001', 'customer': 'Riya', 'date': '2024-06-01', 'sku': 'SKU-B', 'qty': 1, 'price': 1200}
{'order_id': 'ORD002', 'customer': 'Arjun', 'date': '2024-06-01', 'sku': 'SKU-C', 'qty': 5, 'price': 200}


In [12]:
# -----------------------------
# Step 3: Read CSV
# -----------------------------

reconstructed={}

with open(csv_file,"r") as file:
    reader=csv.DictReader(file)
    
    for row in reader:
        order_id = row["order_id"]
        
        if order_id not in reconstructed:
            reconstructed[order_id] = {
                "order_id":order_id,
                "customer":row["customer"],
                "date":row["date"],
                "items":[]
            }
        
        reconstructed[order_id]["items"].append({
            "sku":row["sku"],
            "qty":int(row["qty"]),
            "price":int(row["price"])
        })
        
# Convert dictionary to list
orders_restored=list(reconstructed.values())
# -----------------------------
# Step 4: Print reconstructed JSON
# -----------------------------

from pprint import pprint
print("\nReconstructed JSON:")
pprint(orders_restored)



Reconstructed JSON:
[{'customer': 'Riya',
  'date': '2024-06-01',
  'items': [{'price': 500, 'qty': 2, 'sku': 'SKU-A'},
            {'price': 1200, 'qty': 1, 'sku': 'SKU-B'}],
  'order_id': 'ORD001'},
 {'customer': 'Arjun',
  'date': '2024-06-01',
  'items': [{'price': 200, 'qty': 5, 'sku': 'SKU-C'}],
  'order_id': 'ORD002'}]


In [21]:

# Validate JSON schema Medium
# Write a function validate_record(record, required_keys) that checks if all required keys are present in a dict. Return a dict with valid: True/False and missing_keys: [...]. Run it on each record below.
# required = ["order_id", "customer", "amount", "date"]
# records = [
#     {"order_id": "O1", "customer": "Riya", "amount": 1500, "date": "2024-06-01"},
#     {"order_id": "O2", "amount": 800},
#     {"customer": "Arjun", "date": "2024-06-02"}
# ]

def validate_record(record, required_keys):
    # Find missing keys
    missing_keys = [key for key in required_keys if key not in record]

    return {
        "valid": len(missing_keys) == 0,
        "missing_keys": missing_keys
    }


required = ["order_id", "customer", "amount", "date"]

records = [
    {"order_id": "O1", "customer": "Riya", "amount": 1500, "date": "2024-06-01"},
    {"order_id": "O2", "amount": 800},
    {"customer": "Arjun", "date": "2024-06-02"}
]

# Validate each record
for i, record in enumerate(records, start=1):
    result = validate_record(record, required)
    print(f"Record {i}: {result}")


Record 1: {'valid': True, 'missing_keys': []}
Record 2: {'valid': False, 'missing_keys': ['customer', 'date']}
Record 3: {'valid': False, 'missing_keys': ['order_id', 'amount']}
